# Trend-Filtered Basis Arbitrage: stETH vs. ETH (Liquid Staking Discount)

Same two-idea combination as the companion `paxgGoldArbResearch` notebook — trend detection plus
historical-average deviation arbitrage — applied to a pair with a stronger, better-documented
economic mechanism: **stETH** (Lido's liquid staking token, representing staked ETH + accrued
rewards) vs. **ETH** itself.

**Why this pair specifically**: stETH is a *rebasing* token — staking rewards show up as growing
stETH balance, not price appreciation — so its ETH-denominated exchange rate should trade very close
to 1.0 in calm markets. It doesn't always. During the May-June 2022 UST/Terra collapse and 3AC
contagion, stETH traded at a **real, sustained discount** to ETH (over 6% at the worst point) because
leveraged "staking loop" positions (borrow ETH against stETH collateral, buy more stETH, repeat) got
force-unwound during the crash faster than arbitrageurs could absorb the selling. That's a concrete,
historically-documented mechanism — not a guess — for why ETH's trend might predict the stETH basis:
**forced deleveraging happens during drawdowns, not rallies**, so the discount should widen when ETH
is falling and stay tight (or even flip to a small premium from staking-yield demand) when ETH is
calm or rising.

**Hypothesis, stated directionally** (testable, not assumed): `trend_pct` and `basis_z` should be
**positively correlated** — ETH downtrends coinciding with a widening (more negative) stETH discount,
ETH uptrends coinciding with a tighter/flatter basis. This is a specific, falsifiable prediction we
test in Section 4 before building anything, same as the PAXG notebook.

**Data source**: OKX's public API (free, no key). `STETH-ETH` gives the exchange rate directly — no
need to compute a ratio between two separately-quoted USD legs, which avoids compounding two price
series' noise into one signal.

> ⚠️ Research only, not investment advice. Models a market-neutral spread position (long one leg,
> short the other) with no transaction costs, staking/unstaking queue delays, or execution slippage.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install pandas numpy matplotlib seaborn scipy statsmodels requests

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.5f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data Collection

`STETH-ETH` (the direct exchange rate) and `ETH-USDT` (for a $ price chart and for the trend signal,
computed on ETH itself since that's the asset whose trend we're testing as a predictor).


In [ ]:
OKX_BASE = "https://www.okx.com/api/v5/market"


def fetch_okx_full_history(inst_id, bar="1D"):
    all_rows = []

    r = requests.get(f"{OKX_BASE}/candles", params={"instId": inst_id, "bar": bar, "limit": 300}, timeout=20)
    r.raise_for_status()
    data = r.json()["data"]
    all_rows.extend(data)
    after = min(int(row[0]) for row in data)

    while True:
        r = requests.get(
            f"{OKX_BASE}/history-candles",
            params={"instId": inst_id, "bar": bar, "limit": 100, "after": after},
            timeout=20,
        )
        r.raise_for_status()
        data = r.json()["data"]
        if not data:
            break
        all_rows.extend(data)
        after = min(int(row[0]) for row in data)
        time.sleep(0.25)

    cols = ["ts", "open", "high", "low", "close", "vol", "volCcy", "volCcyQuote", "confirm"]
    df = pd.DataFrame(all_rows, columns=cols)
    df = df[df["confirm"] == "1"]
    df["date"] = pd.to_datetime(df["ts"].astype(np.int64), unit="ms").dt.normalize()
    df = df.drop_duplicates(subset="date").set_index("date").sort_index()
    for c in ["open", "high", "low", "close", "vol"]:
        df[c] = df[c].astype(float)
    return df[["open", "high", "low", "close", "vol"]]


steth_eth = fetch_okx_full_history("STETH-ETH")
eth_usdt = fetch_okx_full_history("ETH-USDT")

print(f"STETH-ETH: {steth_eth.index.min().date()} to {steth_eth.index.max().date()}, {len(steth_eth)} days")
print(f"ETH-USDT:  {eth_usdt.index.min().date()} to {eth_usdt.index.max().date()}, {len(eth_usdt)} days")


## 2. Basis Construction

`STETH-ETH`'s close price *is* the exchange rate — 1.0 means stETH trades exactly at parity with
ETH. `basis_pct` is simply that rate minus 1.


In [ ]:
df = pd.DataFrame({
    "steth_eth_rate": steth_eth["close"],
    "eth_usdt": eth_usdt["close"],
}).dropna()

df["basis_pct"] = df["steth_eth_rate"] - 1

print(f"Overlapping sample: {df.index.min().date()} to {df.index.max().date()}, {len(df)} days")
df.tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df.index, df["eth_usdt"], color="tab:blue")
axes[0].set_title("ETH-USDT Price")

axes[1].plot(df.index, df["basis_pct"] * 100, color="tab:green")
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("stETH/ETH Basis: (STETH-ETH rate - 1) x 100%")
axes[1].set_ylabel("%")
plt.tight_layout()
plt.show()

print(f"Basis stats: mean={df['basis_pct'].mean()*100:.4f}%, std={df['basis_pct'].std()*100:.4f}%, "
      f"min={df['basis_pct'].min()*100:.4f}%, max={df['basis_pct'].max()*100:.4f}%")


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_stat, p_value, *_ = adfuller(df["basis_pct"].dropna())
print(f"ADF test on basis_pct: statistic={adf_stat:.3f}, p-value={p_value:.4f}")
print("p < 0.05 suggests the basis is stationary (mean-reverting) over this sample.")


## 3. Trend Detection (on ETH, the Asset Whose Trend We're Testing as a Predictor)

Same construction as the companion notebook: geometric trend lines through recent swing highs/lows
for visualization, and a rolling regression slope (expressed as an implied %-move-per-window) as the
continuous, backtestable trend signal.


In [ ]:
def find_pivots(price, window=5):
    is_high = price == price.rolling(window * 2 + 1, center=True).max()
    is_low = price == price.rolling(window * 2 + 1, center=True).min()
    return price[is_high], price[is_low]


def fit_trendline(pivot_points, n_recent=3):
    if len(pivot_points) < 2:
        return None
    recent = pivot_points.iloc[-n_recent:]
    x = (recent.index - recent.index[0]).days.values.astype(float)
    y = recent.values
    slope, intercept = np.polyfit(x, y, 1)
    return recent.index[0], slope, intercept


swing_highs, swing_lows = find_pivots(df["eth_usdt"], window=5)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df.index, df["eth_usdt"], color="tab:blue", label="ETH-USDT close", zorder=1)
ax.scatter(swing_highs.index, swing_highs.values, color="red", s=25, zorder=3, label="swing highs")
ax.scatter(swing_lows.index, swing_lows.values, color="green", s=25, zorder=3, label="swing lows")

for pivots, color in [(swing_highs, "red"), (swing_lows, "green")]:
    fit = fit_trendline(pivots, n_recent=4)
    if fit:
        x0, slope, intercept = fit
        x_line = pd.date_range(x0, df.index[-1])
        y_line = intercept + slope * (x_line - x0).days
        ax.plot(x_line, y_line, color=color, linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_title("ETH Price with Recent Swing-High / Swing-Low Trend Lines")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
def rolling_trend_pct(price, window=20):
    def slope(y):
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]
    raw_slope = price.rolling(window).apply(slope, raw=True)
    return raw_slope * window / price

df["trend_pct"] = rolling_trend_pct(df["eth_usdt"], window=20)

fig, ax = plt.subplots()
df["trend_pct"].mul(100).plot(ax=ax, color="purple")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("Rolling 20-Day ETH Trend Signal (Implied %% Move Over Window)")
ax.set_ylabel("%")
plt.tight_layout()
plt.show()


## 4. Does ETH's Trend Predict the stETH Basis? (The Actual Research Question)

Testing the forced-deleveraging hypothesis directly: does `trend_pct` (ETH's rolling trend) correlate
with `basis_z` (the stETH basis's deviation from its own recent average)? The historical mechanism
(2022 depeg during a crash) predicts a **positive** correlation — downtrends coinciding with a wider
discount, uptrends with a tighter one.


In [ ]:
def zscore(s, window=20):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

df["basis_z"] = zscore(df["basis_pct"], window=20)

sub = df[["trend_pct", "basis_z"]].dropna()
corr_val = sub["trend_pct"].corr(sub["basis_z"])

fig, ax = plt.subplots()
ax.scatter(sub["trend_pct"] * 100, sub["basis_z"], alpha=0.5, s=20)
ax.set_xlabel("Rolling ETH trend signal (%% move over 20d)")
ax.set_ylabel("stETH basis z-score")
ax.set_title(f"ETH Trend vs. stETH Basis Deviation (n={len(sub)}, corr={corr_val:.3f})")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
plt.tight_layout()
plt.show()

print(f"Correlation(trend_pct, basis_z) = {corr_val:.3f}")
if abs(corr_val) < 0.15:
    print("Near zero: no meaningful relationship in this sample — a trend filter wouldn't be "
          "expected to help. This sample may simply not contain a real stress event; the 2022 depeg "
          "mechanism is real but episodic, not a constant feature of every period.")
elif corr_val > 0:
    print("Positive, as the forced-deleveraging hypothesis predicts: ETH downtrends coincide with a "
          "wider stETH discount, uptrends with a tighter one.")
else:
    print("Negative — the opposite of the forced-deleveraging hypothesis. Worth investigating why "
          "before leaning on it (e.g. does this sample lack a real stress/liquidation event, making "
          "the relationship dominated by something else entirely?).")


## 5. Mean-Reversion Arbitrage Signal

Same z-score entry/exit state machine as the companion notebook: enter when `|basis_z|` clears the
entry threshold, hold until it reverts inside the exit threshold.

**Position convention**: `+1` = long the basis (long stETH / short ETH, taken when stETH is *cheap*
relative to its own recent history, betting the discount narrows). `-1` = short the basis (short
stETH / long ETH, taken when stETH is unusually rich).


In [ ]:
def build_positions(z, entry=2.0, exit=0.5, blocked=None):
    position = pd.Series(0.0, index=z.index)
    current = 0.0
    for i, (idx, val) in enumerate(z.items()):
        if pd.isna(val):
            position.iloc[i] = current
            continue
        is_blocked = bool(blocked.loc[idx]) if blocked is not None and idx in blocked.index else False
        if current == 0.0:
            if not is_blocked:
                if val > entry:
                    current = -1.0
                elif val < -entry:
                    current = 1.0
        else:
            if abs(val) < exit:
                current = 0.0
        position.iloc[i] = current
    return position


ENTRY_Z, EXIT_Z = 2.0, 0.5
position_pure_mr = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
df["basis_z"].plot(ax=axes[0], color="tab:green")
axes[0].axhline(ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(-ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(0, color="gray", linewidth=0.8)
axes[0].set_title("stETH Basis Z-Score with Entry Thresholds")

position_pure_mr.plot(ax=axes[1], color="black", drawstyle="steps-post")
axes[1].set_title("Pure Mean-Reversion Position (+1 = long basis, -1 = short basis)")
plt.tight_layout()
plt.show()


## 6. Combining Both Ideas: Trend-Filtered Arbitrage

Block new mean-reversion entries while ETH is in a strong trend regime (`|trend_pct|` above its own
adaptive rolling 67th percentile), consistent with the forced-deleveraging hypothesis: don't fade the
discount while ETH is actively crashing (or ramping), since that's exactly when the discount is being
driven by something more persistent than short-term noise.


In [ ]:
trend_strength = df["trend_pct"].abs()
trend_threshold = trend_strength.rolling(90, min_periods=30).quantile(0.67)
strong_trend = trend_strength > trend_threshold

position_trend_filtered = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z, blocked=strong_trend)

print(f"Days blocked by trend filter: {strong_trend.sum()} / {strong_trend.notna().sum()} "
      f"({strong_trend.mean()*100:.1f}%)")


In [ ]:
def sharpe(returns, periods_per_year=365):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)


def max_drawdown(cum_returns):
    running_max = cum_returns.cummax()
    dd = cum_returns / running_max - 1
    return dd.min()


basis_chg = df["basis_pct"].diff()

strategies = pd.DataFrame({
    "pure_mean_reversion": position_pure_mr.shift(1) * basis_chg,
    "trend_filtered_mean_reversion": position_trend_filtered.shift(1) * basis_chg,
})

cum = (1 + strategies.fillna(0)).cumprod()
cum.plot(title="Cumulative Return: Pure vs. Trend-Filtered stETH/ETH Basis Arbitrage")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "annualized_sharpe": strategies.apply(sharpe),
    "total_return": cum.iloc[-1] - 1,
    "annualized_vol": strategies.std() * np.sqrt(365),
    "max_drawdown": cum.apply(max_drawdown),
    "n_trades": pd.Series({
        "pure_mean_reversion": (position_pure_mr.diff().abs() > 0).sum(),
        "trend_filtered_mean_reversion": (position_trend_filtered.diff().abs() > 0).sum(),
    }),
})
summary


## 7. Market-Neutrality Check


In [ ]:
eth_ret = np.log(df["eth_usdt"]).diff()

exposure_check = pd.DataFrame({
    "pure_mr_vs_eth": [strategies["pure_mean_reversion"].corr(eth_ret)],
    "trend_filtered_vs_eth": [strategies["trend_filtered_mean_reversion"].corr(eth_ret)],
}, index=["correlation_with_eth_return"])
exposure_check


## 8. Does the Gross Edge Survive Real Trading Costs?


In [ ]:
TAKER_FEE_PER_FILL = 0.0005
FILLS_PER_ROUND_TRIP = 4

fee_drag = summary["n_trades"] * FILLS_PER_ROUND_TRIP * TAKER_FEE_PER_FILL
net_return_estimate = summary["total_return"] - fee_drag

cost_comparison = pd.DataFrame({
    "gross_total_return": summary["total_return"],
    "est_fee_drag": fee_drag,
    "est_net_total_return": net_return_estimate,
})
cost_comparison


## 9. Takeaways & Next Steps

- **Check Section 4 first**: the forced-deleveraging mechanism is real and historically documented,
  but it's *episodic* — it shows up hardest during genuine stress events (like May-June 2022), not
  necessarily in every arbitrary sample window. If this run's correlation came back weak, that likely
  means the sample simply didn't contain a real stress event, not that the mechanism is wrong.
- **This pair has a structural advantage over PAXG/XAU-perp**: stETH is a rebasing token with a
  precise theoretical peg (1.0, not a market-estimated fair value), and the 2022 depeg gives a much
  stronger prior for *why* trend should predict basis than the PAXG/perp funding-mechanism guess did.
- **Real-world execution isn't modeled**: shorting ETH to hedge a long stETH position requires
  margin/borrow (rates vary and can spike exactly during the stress periods this strategy targets),
  and unstaking stETH back to ETH involves Lido's withdrawal queue (not instant) — a real arbitrageur
  can't necessarily close this trade as fast as the backtest assumes.
- **Extend**: re-run this against a longer history once more OKX data accumulates, and compare
  against other LSTs (rETH, cbETH) to see whether the trend-basis relationship is specific to Lido's
  liquidity/leverage profile or a general liquid-staking phenomenon.
- **Robustness**: validate out-of-sample and account for transaction costs (Section 8) before
  treating any of this as an actual edge.
